In [3]:
!pip install einops

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.6/65.6 kB 2.7 MB/s eta 0:00:00


In [ ]:
import torch
import sys
sys.path.append("/notebooks/segmenter")
from segmenter.segm.model.factory import create_segmenter
from model_v9 import FullModelHAT
import time

# device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True

# MODEL 1
model1 = FullModelHAT(num_classes=11).to(device)
ckpt = torch.load("rescuenet_seg_V13.pth", map_location=device)
model1.load_state_dict(ckpt['model_state'])
model1.eval()

# MODEL 2
model_cfg = {
    "backbone": "vit_small",
    "normalization": "vit",
    "image_size": (713, 713),
    "patch_size": 23,
    "d_model": 512,
    "n_heads": 8,
    "n_layers": 12,
    "distilled": False,
    "decoder": {
        "name": "mask_transformer",
        "n_layers": 2,
        "drop_path_rate": 0.1,
        "dropout": 0.0,
    },
    "n_cls": 11,
}

model2 = create_segmenter(model_cfg).to(device)
model2.eval()


model1 = torch.compile(model1)
# model2 = torch.compile(model2)

# ---------------- DUMMY INPUT ----------------
dummy1 = torch.randn(10, 3, 384, 384).to(device)
dummy2 = torch.randn(10, 3, 713, 713).to(device)

# ---------------- TIMING FUNCTION ----------------
def measure_time(model, name, dummy, runs=100):
    with torch.no_grad():
        for _ in range(50):
            with torch.cuda.amp.autocast():
                _ = model(dummy)

        torch.cuda.synchronize()
        start = time.time()

        for _ in range(runs):
            with torch.cuda.amp.autocast():
                _ = model(dummy)

        torch.cuda.synchronize()
        end = time.time()

    print(f"{name}: {(end-start)/runs*1000:.2f} ms")


# ---------------- RUN ----------------
measure_time(model1, "Model 1 (HAT)", dummy1)
measure_time(model2, "Model 2 (Segmenter)", dummy2)



No pretrained weights exist for this model. Using random initialization.


Model 1 (HAT): 44.10 ms
Model 2 (Segmenter): 49.39 ms
